# MDS650: actividad inusual y RV30

Este notebook es una capa ligera de orquestación/presentación. La fuente de verdad es el paquete local `src/mds650`; no duplica lógica de producción. La pregunta es si la actividad inusual de opciones aporta información incremental fuera de muestra para la varianza realizada de los 30 minutos siguientes. La comparación primaria preespecificada es `Delta_Q = QLIKE(B1) - QLIKE(B2)`.

In [ ]:
# Colab: instalar una revisión etiquetada; local: usar el entorno uv existente.
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # Fije TAG a una versión publicada antes de ejecutar en Colab.
    TAG = os.environ.get('MDS650_REPO_TAG', '001-pit-options-rv30-recovery')
    REPO_URL = os.environ.get('MDS650_REPO_URL')
    if not REPO_URL:
        raise RuntimeError(
            'Set MDS650_REPO_URL before Colab execution'
        )
    get_ipython().system(f'git clone --depth 1 --branch {TAG} {REPO_URL} /content/MDS650-Capstone')
    get_ipython().system('pip install -q /content/MDS650-Capstone')
    REPO = Path('/content/MDS650-Capstone')
else:
    REPO = Path(os.environ.get('MDS650_LOCAL_REPO', Path.cwd()))
    if not (REPO / 'src' / 'mds650').exists():
        raise RuntimeError('Run from the repository or set MDS650_LOCAL_REPO to its root')
sys.path.insert(0, str(REPO / 'src'))
print({'colab': IN_COLAB, 'repo': str(REPO), 'python': sys.version.split()[0]})

In [ ]:
# Cargar Colab Secrets sin imprimir valores; localmente se usan variables de entorno.
from mds650.config import ResearchSettings

if IN_COLAB:
    from google.colab import userdata
    for name in ('UNUSUALWHALES_API_KEY', 'MASSIVE_API_KEY', 'FMP_API_KEY'):
        value = userdata.get(name)
        if value:
            os.environ[name] = value
settings = ResearchSettings()
print({'research_only': settings.research_only, 'secret_presence': settings.secret_presence()})
settings.require_provider_secrets()

In [ ]:
# Configuración congelada de la ejecución autorizada.
FROZEN_CONFIG = {
    'study_window_start': '2025-07-21',
    'study_window_end_exclusive': '2026-07-21',
    'candidate_assets': ['SPY', 'QQQ', 'AAPL', 'MSFT', 'NVDA', 'TSLA', 'AMZN', 'META'],
    'frozen_assets': ['AAPL', 'AMZN', 'MSFT', 'NVDA', 'QQQ', 'SPY'],
    'target': 'RV30',
    'target_prices': 31,
    'target_returns': 30,
    'b1_status': 'INFEASIBLE',
    'fallback_comparison': 'B2-vs-B0',
    'research_only': True,
}
FROZEN_CONFIG

In [ ]:
# Validar el manifiesto de auditoría desde la función modular.
from mds650.manifests import load_and_validate_audit_manifest

manifest_path = (
    REPO / 'artifacts' / 'api_audit' / 'authenticated_v1x'
    / 'provider_audit_manifest.json'
)
validation = load_and_validate_audit_manifest(manifest_path)
print(validation)

In [ ]:
# Mostrar los artefactos congelados y sus conteos; no descarga OPRA ni ejecuta backfill.
import json

run_dir = REPO / 'artifacts' / 'pipeline_runs' / 'window_20260721'
for filename in ('pilot_manifest.json', 'backfill_manifest.json', 'benchmark_manifest.json'):
    payload = json.loads((run_dir / filename).read_text(encoding='utf-8'))
    print(filename, json.dumps(payload, indent=2)[:4000])

In [ ]:
# Esquema observado, controles de calidad y vista previa contractual.
schema_path = (REPO / 'specs' / '001-pit-options-rv30' / 'contracts'
               / 'provider-audit-manifest.schema.json')
schema = json.loads(schema_path.read_text(encoding='utf-8'))
backfill = json.loads((run_dir / 'backfill_manifest.json').read_text(encoding='utf-8'))
print('schema_version:', schema.get('properties', {}).get('schema_version', {}))
print('row_counts_qc:', backfill['row_counts'])
preview = {
    'asset_order': backfill['frozen_assets'],
    'target_contract': backfill['target_contract'],
    'b1_status': backfill['b1_status'],
}
print('preview:', preview)

In [ ]:
# PIT y calidad: el bloqueo de B1 es explícito y no se convierte en una afirmación de valor.
pit_path = (REPO / 'artifacts' / 'api_audit' / 'pit_verification_20260721'
           / 'pit_verification.json')
pit = json.loads(pit_path.read_text(encoding='utf-8'))
pit_keys = ('b1_status', 'ordinary_option_state_pit_verified',
            'fallback_comparison')
print({key: pit.get(key) for key in pit_keys})
print('benchmark_evaluation_authorized:', False)

## Contratos y límites

RV30 usa `C(i,t)` más `C(i,t+1),...,C(i,t+30)` para producir exactamente 30 retornos logarítmicos. Si falta uno de los 31 precios, si la semántica inicio/cierre de FMP no está resuelta o si se requiere interpolación, la fila falla cerradamente. B1 permanece `INFEASIBLE` porque los payloads no prueban una disponibilidad independiente punto-en-tiempo para el estado ordinario de opciones; el fallback B2-vs-B0 está declarado, pero sus métricas no se ejecutan hasta que el contrato de evaluación sea aceptado.